# Notebook 6: Orchestrator-Worker LLM Agent

---

## Learning Objectives

By the end of this notebook, you will be able to:

1. Explain the **Orchestrator-Worker** pattern and how it differs from simple parallelisation.
2. Build an **Orchestrator** that dynamically creates a plan using structured JSON output.
3. Create **specialized Worker chains** with domain-specific system prompts.
4. Build a **Synthesizer** that integrates diverse worker outputs into a cohesive final answer.
5. Run a working **Investment Due Diligence Agent** for comprehensive company analysis.

---

## Use Case: Investment Due Diligence Agent

> **Scenario**: A private equity firm needs to perform due diligence on a company before making an investment decision. The analysis requires multiple types of expertise: financial analysis, competitive strategy, and risk assessment. The Orchestrator-Worker pattern coordinates these specialists automatically.

---

## Orchestrator-Worker vs. Parallelisation

| Feature | Parallelisation | Orchestrator-Worker |
|---|---|---|
| **Task List** | Fixed, predetermined | Dynamically created by Orchestrator |
| **Input per Task** | Same input for all | Different instructions per worker |
| **Coordination** | None needed | Orchestrator plans, Synthesizer integrates |
| **Flexibility** | Low | High |
| **Use Case** | "Analyze this from 3 angles" | "Figure out what needs analyzing, then do it" |

---

## The Architecture

```
┌─────────────────────────────────────────────────────────────────┐
│                         USER REQUEST                            │
│              "Perform due diligence on Tesla, Inc."             │
└──────────────────────────────┬──────────────────────────────────┘
                               │
                               ▼
┌─────────────────────────────────────────────────────────────────┐
│                      ORCHESTRATOR LLM                           │
│                    (gpt-4.1, temp=0.0)                          │
│                                                                 │
│  Analyzes request → Creates structured JSON plan:              │
│  {                                                              │
│    "company": "Tesla",                                          │
│    "subtasks": [                                                │
│      {"task_id": "fin_1", "worker_type": "financial",           │
│       "instructions": "Analyze EV revenue growth..."},         │
│      {"task_id": "moat_1", "worker_type": "moat",              │
│       "instructions": "Evaluate Tesla's brand..."},            │
│      {"task_id": "risk_1", "worker_type": "risk",              │
│       "instructions": "Assess regulatory risks..."}            │
│    ]                                                            │
│  }                                                              │
└──────────────────────────────┬──────────────────────────────────┘
                               │ Dispatches to workers
          ┌────────────────────┼────────────────────┐
          ▼                    ▼                    ▼
┌──────────────────┐ ┌──────────────────┐ ┌──────────────────┐
│  FINANCIAL       │ │  MOAT            │ │  RISK            │
│  WORKER          │ │  WORKER          │ │  WORKER          │
│  (temp=0.3)      │ │  (temp=0.3)      │ │  (temp=0.3)      │
│                  │ │                  │ │                  │
│  Revenue, EPS,   │ │  Competitive     │ │  Regulatory,     │
│  Margins, Debt   │ │  Advantages,     │ │  Market,         │
│                  │ │  Market Share    │ │  Operational     │
└────────┬─────────┘ └────────┬─────────┘ └────────┬─────────┘
         │                    │                    │
         └────────────────────┼────────────────────┘
                              │ All results
                              ▼
┌─────────────────────────────────────────────────────────────────┐
│                      SYNTHESIZER LLM                            │
│                    (gpt-4.1, temp=0.2)                          │
│                                                                 │
│  Integrates all worker reports → Final Investment Memo          │
│  with Investment Verdict (Buy/Hold/Sell)                        │
└─────────────────────────────────────────────────────────────────┘
```

## ⚙️ Step 1: Setup and Azure OpenAI Configuration

We use **three different LLMs** with different roles and temperatures.

In [ ]:
# Install required packages (uncomment if needed)
# !pip install langchain-openai langchain pydantic

import os
from dotenv import load_dotenv
from typing import List, Dict
from pydantic import BaseModel, Field
from langchain_openai import AzureChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser, JsonOutputParser

# ── Load credentials from .env ─────────────────────────────────────────────────
load_dotenv()

# Three LLMs with different roles
orchestrator_llm = AzureChatOpenAI(azure_deployment="gpt-4.1", temperature=0.0)  # Planning: deterministic
worker_llm = AzureChatOpenAI(azure_deployment="gpt-4.1", temperature=0.3)        # Analysis: slightly creative
synthesizer_llm = AzureChatOpenAI(azure_deployment="gpt-4.1", temperature=0.2)   # Writing: balanced

print("Three LLMs initialized:")
print("   Orchestrator: gpt-4.1 (temp=0.0) — deterministic planning")
print("   Workers:      gpt-4.1 (temp=0.3) — analytical reasoning")
print("   Synthesizer:  gpt-4.1 (temp=0.2) — professional writing")

## 🧠 Step 2: Building the Orchestrator

The Orchestrator receives the company name and outputs a structured JSON plan. We use Pydantic to define the exact schema we expect.

In [ ]:
# ── Pydantic Schema for the Orchestrator's Plan ──────────────────────────────
class SubTask(BaseModel):
    task_id: str = Field(description="Unique identifier, e.g., 'fin_1', 'moat_1', 'risk_1'")
    worker_type: str = Field(description="Worker type: must be 'financial', 'moat', or 'risk'")
    instructions: str = Field(
        description="Specific, detailed instructions for this worker tailored to the company's industry"
    )

class DueDiligencePlan(BaseModel):
    company: str = Field(description="The company being analyzed")
    industry: str = Field(description="The company's primary industry")
    subtasks: List[SubTask] = Field(description="List of subtasks for the workers")

# ── Orchestrator Chain ────────────────────────────────────────────────────────
orchestrator_parser = JsonOutputParser(pydantic_object=DueDiligencePlan)

orchestrator_prompt = ChatPromptTemplate.from_messages([
    ("system", """You are a Lead Portfolio Manager at a top-tier investment firm.
    Your job is to plan a comprehensive due diligence process for a company.
    
    Available workers and their expertise:
    - 'financial': Revenue trends, profit margins, debt levels, cash flow, valuation multiples
    - 'moat': Competitive advantages, market share, brand power, network effects, switching costs
    - 'risk': Regulatory risks, competitive threats, operational risks, macroeconomic exposure
    
    Create a plan with exactly 3 subtasks (one per worker type).
    Tailor the instructions to the company's specific industry and business model."""),
    ("user", """Company to analyze: {company}
    
    {format_instructions}""")
]).partial(format_instructions=orchestrator_parser.get_format_instructions())

orchestrator_chain = orchestrator_prompt | orchestrator_llm | orchestrator_parser

print("✅ Orchestrator chain defined.")

## 👷 Step 3: Building the Specialized Workers

Each worker has a domain-specific system prompt. The key insight is that workers receive **different instructions** from the orchestrator — this is what distinguishes this pattern from simple parallelisation.

In [ ]:
def create_worker(role_title: str, expertise_description: str):
    """Factory function to create a specialized worker chain."""
    prompt = ChatPromptTemplate.from_messages([
        ("system", f"""You are a {role_title} at a leading investment bank.
        Your expertise: {expertise_description}
        
        Provide a detailed, analytical report based on your domain expertise.
        Use specific metrics, industry benchmarks, and concrete observations.
        Structure your response with clear sections and bullet points.
        Be concise but insightful — aim for 200-300 words."""),
        ("user", """Company: {company}
        Industry: {industry}
        
        Your specific assignment: {instructions}""")
    ])
    return prompt | worker_llm | StrOutputParser()

# Create three specialized workers
workers = {
    "financial": create_worker(
        role_title="Senior Financial Analyst",
        expertise_description="Income statements, balance sheets, cash flow analysis, DCF valuation, ratio analysis"
    ),
    "moat": create_worker(
        role_title="Strategy Consultant",
        expertise_description="Porter's Five Forces, competitive positioning, brand equity, network effects, switching costs"
    ),
    "risk": create_worker(
        role_title="Chief Risk Officer",
        expertise_description="Regulatory compliance, market risks, operational vulnerabilities, ESG risks, geopolitical exposure"
    )
}

print("✅ Three specialized worker chains created:")
print("   • Financial Analyst — revenue, margins, valuation")
print("   • Strategy Consultant — competitive moat, positioning")
print("   • Chief Risk Officer — regulatory, operational, market risks")

## 📝 Step 4: Building the Synthesizer

The Synthesizer receives all worker reports and writes the final Investment Memo with a clear verdict.

In [ ]:
synthesizer_prompt = ChatPromptTemplate.from_messages([
    ("system", """You are the Managing Director and Lead Portfolio Manager.
    You have received reports from your team of specialists.
    Write a comprehensive, professional Investment Memo for the Investment Committee.
    
    Structure the memo as follows:
    1. Executive Summary (2-3 sentences)
    2. Financial Analysis (key findings from financial worker)
    3. Competitive Position (key findings from moat worker)
    4. Risk Assessment (key findings from risk worker)
    5. Investment Verdict: [BUY / HOLD / SELL] with a clear rationale
    
    Write in a professional, institutional tone."""),
    ("user", """Company: {company}
    Industry: {industry}
    
    SPECIALIST REPORTS:
    {worker_reports}""")
])

synthesizer_chain = synthesizer_prompt | synthesizer_llm | StrOutputParser()

print("✅ Synthesizer chain defined.")

## 🚀 Step 5: The Complete Orchestrator-Worker Pipeline

In [ ]:
def run_due_diligence(company_name: str) -> Dict:
    """
    Run the complete Orchestrator-Worker due diligence pipeline.
    
    Args:
        company_name: The company to analyze
        
    Returns:
        Dictionary with plan, worker_results, and final_memo
    """
    import time
    start = time.time()
    
    # ── Phase 1: Orchestration ────────────────────────────────────────────────
    print(f"\n{'='*60}")
    print(f"🎯 Starting Due Diligence: {company_name}")
    print(f"{'='*60}")
    
    print("\n📋 Phase 1: Orchestrator is planning...")
    plan = orchestrator_chain.invoke({"company": company_name})
    
    print(f"   Company: {plan['company']} | Industry: {plan['industry']}")
    print(f"   Subtasks created: {len(plan['subtasks'])}")
    for st in plan['subtasks']:
        print(f"   • [{st['worker_type'].upper()}] {st['task_id']}: {st['instructions'][:80]}...")
    
    # ── Phase 2: Worker Execution ─────────────────────────────────────────────
    print("\n👷 Phase 2: Workers are analyzing...")
    worker_results = {}
    
    for subtask in plan['subtasks']:
        worker_type = subtask['worker_type']
        print(f"   Running {worker_type} worker...")
        
        result = workers[worker_type].invoke({
            "company": plan['company'],
            "industry": plan['industry'],
            "instructions": subtask['instructions']
        })
        worker_results[subtask['task_id']] = {
            "worker_type": worker_type,
            "report": result
        }
    
    # ── Phase 3: Synthesis ────────────────────────────────────────────────────
    print("\n✍️  Phase 3: Synthesizer is writing the final memo...")
    
    formatted_reports = ""
    for task_id, data in worker_results.items():
        formatted_reports += f"\n--- {data['worker_type'].upper()} ANALYSIS ({task_id}) ---\n"
        formatted_reports += data['report'] + "\n"
    
    final_memo = synthesizer_chain.invoke({
        "company": plan['company'],
        "industry": plan['industry'],
        "worker_reports": formatted_reports
    })
    
    elapsed = time.time() - start
    print(f"\n✅ Due diligence complete in {elapsed:.1f}s")
    
    return {
        "plan": plan,
        "worker_results": worker_results,
        "final_memo": final_memo,
        "elapsed": elapsed
    }

## 🧪 Step 6: Running the Due Diligence Agent

Let's analyze Tesla, Inc. — a company with a fascinating mix of financial complexity, competitive moat, and regulatory risks.

In [ ]:
results = run_due_diligence("Tesla, Inc.")

print("\n" + "="*60)
print("📊 FINAL INVESTMENT MEMO")
print("="*60)
print(results["final_memo"])

## 🧪 Step 7: Testing with a Different Company

Let's run the same pipeline on a completely different company to see how the Orchestrator adapts its plan.

In [ ]:
results2 = run_due_diligence("JPMorgan Chase & Co.")

print("\n" + "="*60)
print("📊 FINAL INVESTMENT MEMO — JPMorgan Chase")
print("="*60)
print(results2["final_memo"])

## 📚 Summary and Key Takeaways

| Component | Role | Temperature |
|---|---|---|
| **Orchestrator** | Plans the analysis — creates subtasks with specific instructions | 0.0 (deterministic) |
| **Financial Worker** | Analyzes revenue, margins, valuation | 0.3 (analytical) |
| **Moat Worker** | Evaluates competitive advantages | 0.3 (analytical) |
| **Risk Worker** | Identifies threats and vulnerabilities | 0.3 (analytical) |
| **Synthesizer** | Writes the final Investment Memo | 0.2 (professional writing) |

### 🔑 Key Insights

1. **Dynamic vs. Static**: Unlike Parallelisation (Notebook 5), the Orchestrator creates different instructions for each worker based on the specific company. Tesla's financial analysis instructions differ from JPMorgan's.
2. **Specialization produces depth**: Each worker's focused system prompt produces much deeper analysis than a single generalist prompt.
3. **Scalability**: To add a new analysis dimension (e.g., ESG analysis), simply add a new worker type and update the Orchestrator's prompt — no other changes needed.
4. **The Synthesizer is crucial**: Raw worker outputs are valuable but fragmented. The Synthesizer transforms them into a coherent, actionable document.

### 🔄 Comparison: All 6 Patterns

| Pattern | Notebook | Key Mechanism | Best For |
|---|---|---|---|
| ReAct Agent | 1 | Think-Act-Observe loop | Tool use, external data |
| Persistent Memory | 2 | Dual-track retrieve+update | Personalized conversations |
| Prompt Chaining | 3 | Sequential `\|` composition | Multi-step analysis |
| Evaluator-Optimiser | 4 | Generate-Evaluate-Revise loop | Quality-critical outputs |
| Parallelisation | 5 | Same input, multiple chains | Fast multi-perspective analysis |
| Orchestrator-Worker | 6 | Dynamic planning + specialization | Complex, multi-domain tasks |
